# Ant Colony Optimisation: a six-stop delivery route

This worked example follows one ACO iteration for a small delivery route. The stops are Depot, Market, Campus, Station, Mall and Park. We calculate move probabilities, score three tours, update pheromone, then compare the next-move probabilities. The tours are fixed here so each calculation is easy to follow; a full ACO run would sample new tours and repeat.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Distance matrix and settings

Distances are symmetric, and the diagonal is zero. `alpha` controls the influence of pheromone, `beta` weights the distance heuristic, `rho` is the evaporation rate, and `Q` sets the amount deposited.

In [ ]:
stops = np.array(["Depot", "Market", "Campus", "Station", "Mall", "Park"])
distance = np.array([
    [0, 4, 7, 9, 6, 8],
    [4, 0, 5, 6, 3, 7],
    [7, 5, 0, 4, 6, 3],
    [9, 6, 4, 0, 5, 4],
    [6, 3, 6, 5, 0, 5],
    [8, 7, 3, 4, 5, 0]
], dtype=float)
alpha, beta, rho, Q = 1, 2, 0.4, 90
pheromone = np.ones_like(distance)
np.fill_diagonal(pheromone, 0)

def show_matrix(matrix, title):
    print("\n" + title)
    print("".join(f"{s:>10}" for s in stops))
    for name, row in zip(stops, matrix):
        print(f"{name:>10}" + "".join(f"{v:10.3f}" for v in row))

show_matrix(distance, "Distance matrix")
show_matrix(pheromone, "Initial pheromone")

## 2. Desirability and transition probabilities

The heuristic is the inverse of distance, \(\eta_{ij}=1/d_{ij}\). For each eligible next stop, calculate \(w_{ij}=\tau_{ij}^{\alpha}\eta_{ij}^{\beta}\), then divide by the sum of eligible weights. Stops already visited are excluded.

In [ ]:
eta = np.zeros_like(distance)
np.divide(1.0, distance, out=eta, where=distance > 0)

def probabilities(current, candidates, trail=pheromone, show=True):
    candidates = np.asarray(candidates, dtype=int)
    weights = trail[current, candidates]**alpha * eta[current, candidates]**beta
    probs = weights / weights.sum()
    if show:
        print(f"From {stops[current]} (weight sum {weights.sum():.5f})")
        for j, w, p in zip(candidates, weights, probs):
            print(f"  {stops[j]:>8}: weight={w:.5f}, p={p:.4f}")
    return probs

for start in [0, 1, 2]:
    print("\nFirst move")
    probabilities(start, [j for j in range(len(stops)) if j != start])

## 3. Tours for this iteration

These are the three tours used for the worked calculation. Each visits every stop once and returns to its starting point. In an algorithm run, the move probabilities guide stochastic selection rather than forcing the same tours every time.

In [ ]:
tours = [
    [0, 1, 4, 3, 2, 5, 0],
    [1, 0, 5, 2, 3, 4, 1],
    [2, 5, 3, 4, 1, 0, 2]
]
for k, tour in enumerate(tours, 1):
    print(f"Ant {k}: " + " -> ".join(stops[i] for i in tour))

# Show the available choices at each move; pheromone is unchanged until all tours finish.
for k, tour in enumerate(tours, 1):
    print(f"\nANT {k}")
    visited = {tour[0]}
    for i, j in zip(tour[:-2], tour[1:-1]):
        candidates = [x for x in range(len(stops)) if x not in visited]
        probabilities(i, candidates)
        print(f"  Recorded move: {stops[i]} -> {stops[j]}")
        visited.add(j)
    print(f"  Return: {stops[tour[-2]]} -> {stops[tour[-1]]}")

## 4. Tour lengths and deposits

The tour length includes the return edge: \(L_k=\sum d_{ij}\). Each ant deposits \(Q/L_k\) on every edge in its tour, so shorter tours contribute more pheromone.

In [ ]:
lengths = []
for k, tour in enumerate(tours, 1):
    edge_costs = distance[tour[:-1], tour[1:]]
    total = edge_costs.sum()
    lengths.append(total)
    print(f"Ant {k}: " + " + ".join(f"{x:g}" for x in edge_costs) + f" = {total:g}")
lengths = np.asarray(lengths)
rewards = Q / lengths
for k, (length, reward) in enumerate(zip(lengths, rewards), 1):
    print(f"Ant {k}: deposit per used edge = {Q}/{length:g} = {reward:.4f}")

## 5. Combine deposits and evaporate old trails

Deposits from ants using the same edge are added. Since the route is undirected, each deposit is entered in both directions. Evaporation leaves \((1-\rho)\tau_{ij}\) of the old pheromone.

In [ ]:
deposits = np.zeros_like(distance)
for tour, reward in zip(tours, rewards):
    for i, j in zip(tour[:-1], tour[1:]):
        deposits[i, j] += reward
        deposits[j, i] += reward

evaporated = (1 - rho) * pheromone
updated = evaporated + deposits
show_matrix(deposits, "Combined deposits")
show_matrix(evaporated, "After evaporation")
show_matrix(updated, "Updated pheromone: evaporation + deposits")

## 6. What changes for the next move?

The distances stay the same, but the pheromone matrix changes. The bars compare the probability of leaving Depot for each other stop before and after the update.

In [ ]:
candidates = [1, 2, 3, 4, 5]
before = probabilities(0, candidates, pheromone, show=False)
after = probabilities(0, candidates, updated, show=False)
for j, p0, p1 in zip(candidates, before, after):
    print(f"Depot -> {stops[j]}: {p0:.4f} -> {p1:.4f}")
x = np.arange(len(candidates))
plt.figure(figsize=(8, 4))
plt.bar(x - 0.18, before, 0.36, label="Before update")
plt.bar(x + 0.18, after, 0.36, label="After update")
plt.xticks(x, [stops[j] for j in candidates])
plt.ylabel("Transition probability")
plt.xlabel("Possible next stop from Depot")
plt.title("Effect of the pheromone update")
plt.legend()
plt.tight_layout()
plt.show()

best = int(np.argmin(lengths))
print("\nBest of these tours:", " -> ".join(stops[i] for i in tours[best]))
print("Tour lengths:", lengths.tolist())

## Conclusion

The update has four parts: calculate the full tour cost, reward edges in proportion to \(Q/L_k\), evaporate old pheromone, and add the new deposits. Those updated trail values affect the next set of move probabilities. A single iteration compares only the tours used here; repeated iterations are needed to search more broadly for a short route.